# Question B2 (10 marks)
In Question B1, we used the Category Embedding model. This creates a feedforward neural network in which the categorical features get learnable embeddings. In this question, we will make use of a library called Pytorch-WideDeep. This library makes it easy to work with multimodal deep-learning problems combining images, text, and tables. We will just be utilizing the deeptabular component of this library through the TabMlp network:

In [3]:
# !pip install pytorch-widedeep

In [18]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

import torch

from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.models import TabMlp, WideDeep
from pytorch_widedeep import Trainer
from pytorch_widedeep.metrics import R2Score

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

## Configuration for Google Colab
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/SC4001 Neural Network')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1.Divide the dataset (‘hdb_price_prediction.csv’) into train and test sets by using entries from the year 2020 and before as training data, and entries from 2021 and after as the test data.

In [5]:
df = pd.read_csv('/content/drive/MyDrive/SC4001 Neural Network/hdb_price_prediction.csv')

# YOUR CODE HERE

# Filter Out data from year 2022 and 2023
# Define features that should be used
numeric_features     = ['dist_to_nearest_stn', 'dist_to_dhoby', 'degree_centrality', 'eigenvector_centrality', 'remaining_lease_years', 'floor_area_sqm']
categorical_features = ['month', 'town', 'flat_model_type', 'storey_range']
prediction_target    = ['resale_price']
all_features         = numeric_features + categorical_features + prediction_target

# Split features for train, validation & test by year
train_df = df[df['year'] <= 2020]
test_df  = df[df['year'] >= 2021]

# Separate features and target
train_df = train_df[all_features]
test_df  = test_df[all_features]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
print("Train Data Shape:", train_df.shape)
print("Test Data Shape:", test_df.shape)

Train Data Shape: (87370, 11)
Test Data Shape: (72183, 11)


2.Refer to the documentation of Pytorch-WideDeep and perform the following tasks:
https://pytorch-widedeep.readthedocs.io/en/latest/index.html
* Use [**TabPreprocessor**](https://pytorch-widedeep.readthedocs.io/en/latest/examples/01_preprocessors_and_utils.html#2-tabpreprocessor) to create the deeptabular component using the continuous
features and the categorical features. Use this component to transform the training dataset.
* Create the [**TabMlp**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/model_components.html#pytorch_widedeep.models.tabular.mlp.tab_mlp.TabMlp) model with 2 linear layers in the MLP, with 250 and 150 neurons respectively.
* Create a [**Trainer**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/trainer.html#pytorch_widedeep.training.Trainer) for the training of the created TabMlp model with the mean squared error (MSE) cost function. Train the model for 80 epochs using this trainer, keeping a batch size of 64. (Note: set the *num_workers* parameter to 0.)

In [7]:
# YOUR CODE & RESULT HERE
tab_preprocessor = TabPreprocessor(
    cat_embed_cols = categorical_features,
    continuous_cols = numeric_features
)

X_tab = tab_preprocessor.fit_transform(train_df)

print(X_tab.shape)

/usr/local/lib/python3.12/dist-packages/pytorch_widedeep/preprocessing/tab_preprocessor.py:364: UserWarning: Continuous columns will not be normalised
  warnings.warn("Continuous columns will not be normalised")


(87370, 10)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
tab_mlp = TabMlp(
    column_idx = tab_preprocessor.column_idx,
    cat_embed_input = tab_preprocessor.cat_embed_input,
    continuous_cols = numeric_features,
    mlp_hidden_dims = [250, 150]
)

model = WideDeep(deeptabular = tab_mlp)

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [10]:
trainer = Trainer(
    model,
    objective = 'mse',
    num_workers = 0,
    seed = SEED,
    device = device
)

# Train the model
trainer.fit(
    X_tab = X_tab,
    target = train_df['resale_price'].values,
    n_epochs = 80,
    batch_size = 64,
)

epoch 80: 100%|██████████| 1366/1366 [00:15<00:00, 86.32it/s, loss=1.98e+9]


3.Report the test MSE and the test R2 value that you obtained.

In [20]:
# YOUR CODE & RESULT HERE
X_tab_test = tab_preprocessor.transform(test_df)

# Get predictions
predicted = trainer.predict(X_tab = X_tab_test, batch_size = 64)
actual = test_df['resale_price']

r2 = r2_score(actual, predicted)
mse = mean_squared_error(actual, predicted)

print(f"\nTest MSE: {mse:.4f}")
print(f"Test R^2: {r2:.4f}\n")

  0%|          | 0/1128 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
predict: 100%|██████████| 1128/1128 [00:03<00:00, 294.49it/s]



Test MSE: 9960677875.1189
Test R^2: 0.6520



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
